Run `python run_pipeline.py` from the project root (or execute each model
notebook once, in any order) so every `metrics_*.csv` lands in `../results/`,
then run this notebook. It reads those files and writes
`comparison_summary.csv` and `best_model_by_window.csv` back to `../results/`.

In [4]:
import pandas as pd

In [1]:
MODEL_FILES = {
    "VECM": "../results/metrics_var_vecm.csv",
    "ARIMA": "../results/metrics_arima.csv",
    "LSTM": "../results/metrics_lstm.csv",
    "TDNN": "../results/metrics_tdnn.csv",
    "FFNN": "../results/metrics_ffnn.csv",
    "Hybrid_Residual": "../results/metrics_hybrid_residual.csv",
    "Hybrid_Neural": "../results/metrics_hybrid_neural.csv",
    "Ensemble_Simple": "../results/metrics_hybrid_ensemble_simple.csv",
    "Ensemble_Weighted": "../results/metrics_hybrid_ensemble_weighted.csv",
    "Ensemble_Stacked": "../results/metrics_hybrid_ensemble_stacked.csv",
}
WINDOW_COLS = ["W1", "W2", "W3", "W4", "W5"]


In [5]:
import os

def load_metrics(model: str, path: str) -> pd.DataFrame:
    """Load one model's standardized long-format export: window, target, RMSE, MAE."""
    df = pd.read_csv(path)
    df.insert(0, "model", model)
    return df


available = {}
for model, path in MODEL_FILES.items():
    if os.path.exists(path):
        available[model] = load_metrics(model, path)
    else:
        print(f"[skip] {path} not found — excluding {model} from the comparison")

long_df = pd.concat(available.values(), ignore_index=True)
long_df


,model,window,target,RMSE,MAE
0,VECM,W1,CPI,4.605201,3.972368
1,VECM,W1,EXR,4.663703,4.266680
2,VECM,W2,CPI,2.210647,1.900296
3,VECM,W2,EXR,5.841530,5.379869
4,VECM,W3,CPI,6.516029,5.157529
...,...,...,...,...,...
95,Ensemble_Stacked,W3,EXR,8.610959,8.016288
96,Ensemble_Stacked,W4,CPI,3.042829,3.037297
97,Ensemble_Stacked,W4,EXR,7.365443,6.954511
98,Ensemble_Stacked,W5,CPI,8.252797,5.568296


In [6]:
# Rows: model x target x metric. Columns: W1..W5.
comparison = (
    long_df
    .melt(id_vars=["model", "window", "target"], value_vars=["RMSE", "MAE"], var_name="metric")
    .pivot_table(index=["model", "target", "metric"], columns="window", values="value")
    .reindex(columns=WINDOW_COLS)
    .reindex(MODEL_FILES.keys(), level="model")
)
comparison.round(4)

window                                W1       W2       W3       W4        W5
model             target metric                                              
VECM              CPI    MAE      3.9724   1.9003   5.1575  22.1381   38.1843
                         RMSE     4.6052   2.2106   6.5160  23.2167   41.7019
                  EXR    MAE      4.2667   5.3799   7.8951  12.0885  106.2636
                         RMSE     4.6637   5.8415   9.9994  14.6161  124.5505
ARIMA             CPI    MAE      7.3312   3.5954  11.6650   5.0124   13.9772
                         RMSE     8.6187   4.2849  15.2640   5.1161   17.2782
                  EXR    MAE      3.1369   8.2943   0.5837   2.3589   35.7643
                         RMSE     3.4391   9.0520   0.6432   2.4349   39.3133
LSTM              CPI    MAE     10.2457   1.6649  10.5894   2.0346    8.0564
                         RMSE    10.8902   2.3240  13.6506   2.3010   11.7554
                  EXR    MAE      2.8295   5.6079   1.8837   6.7517   15.7182
                         RMSE     3.0760   5.8992   2.3776   6.8582   21.3244
TDNN              CPI    MAE      7.8650   1.9748   4.4408   3.1020   48.1173
                         RMSE     8.9941   2.3240   7.1996   3.6890   55.0373
                  EXR    MAE      2.5837   8.9111  11.1150   4.0130   40.0799
                         RMSE     2.7521  10.0683  13.8351   5.0184   56.2680
FFNN              CPI    MAE      5.1803   6.8634  14.8074   2.9651   47.5398
                         RMSE     5.6309   7.1902  16.8730   3.0421   48.3946
                  EXR    MAE      5.0854   8.1720   8.8783   7.5380    9.6391
                         RMSE     5.1534   8.1970   9.4218   8.0445   10.1853
Hybrid_Residual   CPI    MAE      7.1116   3.3371  11.3228   5.4401   14.0433
                         RMSE     8.4104   3.9887  14.9929   5.5580   17.3708
                  EXR    MAE      3.2707   8.1675   0.5779   1.9353   35.9185
                         RMSE     3.6254   8.8953   0.7321   1.9961   39.5351
Hybrid_Neural     CPI    MAE      8.9873   2.2682  15.1602   1.7898  105.0917
                         RMSE    11.1490   3.0836  19.4094   2.3170  115.7290
                  EXR    MAE      1.5547   6.0509  10.7474   8.5722   98.0421
                         RMSE     1.7565   7.4815  11.0827   9.3274  110.9694
Ensemble_Simple   CPI    MAE      8.7817   1.9755  11.6902   3.0373   13.2834
                         RMSE     9.1453   2.5665  13.8311   3.0428   17.4027
                  EXR    MAE      4.0100   8.6620   8.0163   6.9545   23.4007
                         RMSE     4.0421   8.6751   8.6110   7.3654   27.1185
Ensemble_Weighted CPI    MAE      7.8368   1.5375  10.0820   1.1404    9.6449
                         RMSE     8.1979   1.8690  12.2054   1.2762   11.0893
                  EXR    MAE      3.7403   8.1183   1.6074   3.4060   11.7507
                         RMSE     3.7780   8.1352   1.7922   3.7093   13.0610
Ensemble_Stacked  CPI    MAE      8.7817   1.9755  11.6902   3.0373    5.5683
                         RMSE     9.1453   2.5665  13.8311   3.0428    8.2528
                  EXR    MAE      4.0100   8.6620   8.0163   6.9545    6.4310
                         RMSE     4.0421   8.6751   8.6110   7.3654    7.6194

In [7]:
# Best-performing model per window/target, ranked by RMSE.
best_by_window = (
    long_df.loc[long_df.groupby(["window", "target"])["RMSE"].idxmin(), ["window", "target", "model", "RMSE"]]
    .set_index(["window", "target"])
    .sort_index()
)
best_by_window

model      RMSE
window target                             
W1     CPI                  VECM  4.605201
       EXR         Hybrid_Neural  1.756537
W2     CPI     Ensemble_Weighted  1.868965
       EXR                  VECM  5.841530
W3     CPI                  VECM  6.516029
       EXR                 ARIMA  0.643180
W4     CPI     Ensemble_Weighted  1.276187
       EXR       Hybrid_Residual  1.996118
W5     CPI      Ensemble_Stacked  8.252797
       EXR      Ensemble_Stacked  7.619369

In [8]:
# Persist the cross-model comparison so it survives outside this notebook.
comparison.round(4).to_csv("../results/comparison_summary.csv")
best_by_window.to_csv("../results/best_model_by_window.csv")
